# Relaxed heuristics for A*

_Artificial Intelligence — more_

**Drop a rule to make an easier problem. Its cost becomes a heuristic that never lies high.**

A* needs a heuristic $h(s)$: a guess of the remaining cost to the goal. It must never overestimate.

     Where do we get such a guess? Relax the problem: throw away a constraint to make an easier version.

---

This notebook is a practice scaffold for the **Relaxed heuristics for A*** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

In [ ]:
import numpy as np
from collections import deque

# 5x5 grid; 1 = wall. Start (0,0), goal (4,4).
grid = np.zeros((5, 5), int)
for r, c in [(1,0),(1,1),(1,2),(1,3),(3,1),(3,2),(3,3),(3,4)]:
    grid[r, c] = 1
start, goal = (0, 0), (4, 4)

def manhattan(a, b):                      # relaxed cost: walls removed
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def true_cost(grid, s, g):                # BFS shortest path obeying walls
    q, seen = deque([(s, 0)]), {s}
    while q:
        (r, c), d = q.popleft()
        if (r, c) == g:
            return d
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < 5 and 0 <= nc < 5 and grid[nr, nc] == 0 and (nr, nc) not in seen:
                seen.add((nr, nc)); q.append(((nr, nc), d+1))
    return float("inf")

h, cost = manhattan(start, goal), true_cost(grid, start, goal)
print("relaxed heuristic h =", h)
print("true maze cost      =", cost)
print("admissible (h <= cost)?", h <= cost)

## Visualize the data & results

_Hospital delivery robot: how good is the relaxed (walls-removed) A* heuristic for the route to the Pharmacy?_

In [ ]:
import numpy as np, heapq
import matplotlib.pyplot as plt

# Hospital 1st-floor wing as a 6x6 grid. Goal = Pharmacy at (5,5).
# Walls form a serpentine corridor between wards, labs and the pharmacy.
GRID, goal = 6, (5, 5)
walls = {(0,1),(1,1),(2,1),(3,1),(4,1),
         (1,3),(2,3),(3,3),(4,3),(5,3),
         (0,5),(1,5),(2,5),(3,5)}

# Relaxed heuristic: Manhattan steps to the Pharmacy with walls removed.
H = np.zeros((GRID, GRID), int)
for r in range(GRID):
    for c in range(GRID):
        H[r, c] = abs(r - goal[0]) + abs(c - goal[1])

# Real A* on the walled grid confirms h(start)=10 vs true cost=20 (admissible).
def astar(start):
    pq, seen = [(H[start], 0, start)], {}
    while pq:
        f, g, s = heapq.heappop(pq)
        if s in seen and g >= seen[s]: continue
        seen[s] = g
        if s == goal: return g
        r, c = s
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            inb = nr in range(GRID) and nc in range(GRID)
            if inb and (nr, nc) not in walls:
                heapq.heappush(pq, (g+1+H[nr,nc], g+1, (nr,nc)))
print("h(start)=", H[0,0], " true walled cost=", astar((0,0)))

fig, ax = plt.subplots()
im = ax.imshow(H, cmap="viridis")
for r in range(GRID):
    for c in range(GRID):
        ax.text(c, r, H[r, c], ha="center", va="center", color="w")
ax.set_title("Relaxed heuristic h = steps to Pharmacy (5,5), walls ignored")
fig.colorbar(im, ax=ax)
plt.show()